# IVF Insight — Dataset Audit and Exploratory Analysis

This notebook is a reproducible research workflow. It does **not** establish clinical validity or recommend treatment. Do not load identifiable patient data.

**Before modelling:** verify dataset version/licence, data dictionary, target definition, cohort unit, and which variables are available at the intended prediction time. Exclude identifiers and post-outcome variables.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('../data/raw')
files = sorted([*DATA_DIR.glob('*.csv'), *DATA_DIR.glob('*.xlsx'), *DATA_DIR.glob('*.xls')])
if not files:
    raise FileNotFoundError('Place the licensed dataset file in data/raw/ and rerun this cell.')
DATA_PATH = files[0]
print('Selected file:', DATA_PATH.name)
sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print('SHA-256:', sha256)
df = pd.read_csv(DATA_PATH) if DATA_PATH.suffix.lower() == '.csv' else pd.read_excel(DATA_PATH)
print('Shape:', df.shape)


## 1. Schema and data dictionary review

Review the printed column names and definitions against the dataset's official documentation. Do not infer clinical meaning from a column name alone.

In [ ]:
schema = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing_n': [int(df[c].isna().sum()) for c in df.columns],
    'missing_pct': [round(float(df[c].isna().mean()*100), 2) for c in df.columns],
    'unique_n': [int(df[c].nunique(dropna=True)) for c in df.columns],
})
display(schema)
display(df.head(3))  # Inspect locally only; never publish patient-level rows


## 2. Data-quality audit

Check duplicates, missingness, impossible values, target prevalence, and whether multiple rows belong to the same patient/cycle. The dataset description says values were normalised and categorical variables encoded; verify this before interpreting units.

In [ ]:
print('Duplicate rows:', int(df.duplicated().sum()))
display(df.describe(include='all').T)
missing = schema.sort_values('missing_pct', ascending=False)
ax = missing.plot.bar(x='column', y='missing_pct', legend=False, figsize=(12,4), title='Missingness by column')
ax.set_ylabel('Missing (%)'); plt.xticks(rotation=70, ha='right'); plt.tight_layout(); plt.show()


## 3. Define the outcome and prediction time

Set `TARGET` to the exact outcome column. Then explicitly list variables unavailable at the intended prediction time, identifiers, and potential leakage variables. Do not continue to modelling until this review is complete with the data dictionary.

In [ ]:
TARGET = 'Output(failed implantation or successful implantation)' if 'Output(failed implantation or successful implantation)' in df.columns else ('Live birth or not' if 'Live birth or not' in df.columns else df.columns[-1])
if TARGET not in df.columns:
    raise KeyError(f'Update TARGET. Available columns: {df.columns.tolist()}')

print('Outcome distribution:')
display(df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame('n'))
print('Outcome proportions:')
display(df[TARGET].value_counts(normalize=True, dropna=False).rename('proportion'))


## 4. Exploratory analysis

Only after confirming column meanings, examine numeric distributions and relationships. These are exploratory associations, not causal effects. Account for multiple comparisons and avoid presenting p-values as proof of clinical importance.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    df[numeric_cols].hist(figsize=(14, 10), bins=20)
    plt.suptitle('Numeric feature distributions')
    plt.tight_layout(); plt.show()

corr = df[numeric_cols].corr(numeric_only=True)
display(corr.round(2))


## 5. Research decision log

Before modelling, record: (1) target meaning and positive class, (2) unit of analysis, (3) prediction time, (4) patient/cycle grouping, (5) excluded identifiers and post-outcome variables, (6) source version/licence, and (7) limitations. If multiple cycles per patient exist, a random row split can leak patient-specific information; use group-aware validation where appropriate.

In [ ]:
# Save a privacy-safe schema summary; no individual rows are exported.
out = Path('../data/processed'); out.mkdir(parents=True, exist_ok=True)
schema.to_csv(out / 'schema_audit.csv', index=False)
(Path('../data/processed/dataset_sha256.txt')).write_text(sha256 + '  ' + DATA_PATH.name + '\n')
print('Saved schema audit and dataset checksum.')
